In [1]:
import numpy as np
import pandas as pd
from scipy.stats import qmc 
from scipy.interpolate import interp1d 
from tqdm import tqdm
import time

import sys, os, pathlib

NOTEBOOK_DIR = pathlib.Path(os.getcwd())  
ROOT = NOTEBOOK_DIR.parents[0]  
sys.path.insert(0, str(ROOT))
from src.Heston_FFT import heston_price_fft
from src.dataset_generator import sample_heston_lhs


# params
n_samples = 30_000

alpha = 1.5
n = 4096
eta = 0.1


df_first = sample_heston_lhs(n_samples, seed=123, S0 = 105, q=0)
df_first.head()


,S0,K,m,T,r,q,v0,kappa,theta,sigma_v,rho
0,105,107.154867,1.020523,0.721133,0.061152,0,0.058228,3.203556,0.030715,0.359458,-0.236159
1,105,111.880378,1.065527,0.109308,-0.005739,0,0.041815,2.916488,0.047404,0.279555,-0.680496
2,105,70.661150,0.672963,0.720562,0.080410,0,0.058371,3.535999,0.077431,0.217583,-0.621342
3,105,101.879992,0.970286,0.125118,0.057108,0,0.068416,1.875307,0.068807,0.357617,-0.117508
4,105,80.905708,0.770531,0.978733,-0.005538,0,0.047923,2.109171,0.076900,0.475892,-0.199244


In [2]:
def price_heston_fft_df(
    df: pd.DataFrame,
    option_type: str = "call",
    alpha: float = 1.5,
    N: int = 4096,
    eta: float = 0.05,
) -> pd.DataFrame:
    """
    Adds the 'price' columns via FFT

    Required cols:
    ['S0','K','T','r','q','v0','kappa','theta','sigma_v','rho'].
    """
    required = ["S0","K","T","r","q","v0","kappa","theta","sigma_v","rho"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing cols: {missing}")

    prices = []
    for _, row in df.iterrows():
        try:
            p = heston_price_fft(
                S=float(row["S0"]),
                K=float(row["K"]),
                T=float(row["T"]),
                r=float(row["r"]),
                v0=float(row["v0"]),
                kappa=float(row["kappa"]),
                theta=float(row["theta"]),
                sigma_v=float(row["sigma_v"]),
                rho=float(row["rho"]),
                option_type=option_type,
                q=float(row["q"]),
                alpha=alpha,
                N=N,
                eta=eta,
            )
        except Exception:
            p = np.nan
        prices.append(p)

    out = df.copy()
    out["price"] = prices
    return out


df = price_heston_fft_df(df_first)
df.head()

,S0,K,m,T,r,q,v0,kappa,theta,sigma_v,rho,price
0,105,107.154867,1.020523,0.721133,0.061152,0,0.058228,3.203556,0.030715,0.359458,-0.236159,8.319422
1,105,111.880378,1.065527,0.109308,-0.005739,0,0.041815,2.916488,0.047404,0.279555,-0.680496,0.568440
2,105,70.661150,0.672963,0.720562,0.080410,0,0.058371,3.535999,0.077431,0.217583,-0.621342,38.581749
3,105,101.879992,0.970286,0.125118,0.057108,0,0.068416,1.875307,0.068807,0.357617,-0.117508,6.027521
4,105,80.905708,0.770531,0.978733,-0.005538,0,0.047923,2.109171,0.076900,0.475892,-0.199244,25.726334


In [3]:
# random check
price_fft_example = heston_price_fft(S=105, K=74.4210, T=0.4142, r=0.0770, v0=0.0352, kappa=1.2193, theta=0.0685, sigma_v=0.3550, rho=-0.6053,
                             option_type="call", q=0.0, alpha=alpha, N=n, eta=eta)

price_fft_example

32.999084677497365

In [4]:
df.drop(columns=['q', 'S0', 'K'], inplace = True)
df

,m,T,r,v0,kappa,theta,sigma_v,rho,price
0,1.020523,0.721133,0.061152,0.058228,3.203556,0.030715,0.359458,-0.236159,8.319422
1,1.065527,0.109308,-0.005739,0.041815,2.916488,0.047404,0.279555,-0.680496,0.568440
2,0.672963,0.720562,0.080410,0.058371,3.535999,0.077431,0.217583,-0.621342,38.581749
3,0.970286,0.125118,0.057108,0.068416,1.875307,0.068807,0.357617,-0.117508,6.027521
4,0.770531,0.978733,-0.005538,0.047923,2.109171,0.076900,0.475892,-0.199244,25.726334
...,...,...,...,...,...,...,...,...,...
29995,0.932392,1.317061,0.004498,0.042941,3.231626,0.047553,0.198924,-0.860998,14.338963
29996,1.261402,0.348496,0.049573,0.056867,1.890238,0.054830,0.297474,-0.219644,0.390890
29997,1.240718,0.943924,0.050631,0.048016,4.980503,0.048860,0.161180,-0.878109,2.671822
29998,0.911696,0.679220,0.095922,0.074944,2.576565,0.026691,0.358211,-0.503763,17.535772


In [ ]:
import os
os.makedirs("data", exist_ok=True)
#df.to_csv("data/df_train_nn.csv", index=False)